In [1]:
#Project set up

#All necessary packages
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import CelebA
from torchvision.models import resnet18, ResNet18_Weights
from torchmetrics.classification import BinaryAccuracy, BinaryAUROC, BinaryF1Score

# Setting seed for reproducability
torch.manual_seed(42)

# Device configuration - will choose CPU for my setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active execution device: {device}")

# Global Hyperparameters
BATCH_SIZE = 64        # Number of training samples passed through the neural network on each forward/backward pass
LEARNING_RATE = 1e-4    # Use smaller learning rate as ResNet is pretrained. Default range for Adam Opt
NUM_EPOCHS = 5          # Number of training passes per part
DATA_DIR = "./data"     

# Target label
TARGET_ATTR = "Smiling"

# 10 Facial Attributes of Interest
CONCEPT_ATTRS = [
    "Mouth_Slightly_Open",
    "High_Cheekbones",
    "Chubby",
    "Narrow_Eyes",
    "Bags_Under_Eyes",
    "Big_Lips",
    "Big_Nose",
    "Pointy_Nose",
    "Bushy_Eyebrows",
    "Arched_Eyebrows",
]

# variable for concept quantity
NUM_CONCEPTS = len(CONCEPT_ATTRS) 

# Crops the image to reduce pixels for compute time and makes it square
transform = transforms.Compose([
    transforms.CenterCrop(178), #Crop first so PyTorch doesn't squish rectangular image into a square with Resize
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

# Load our datasets
full_train_ds = CelebA(root=DATA_DIR, split="train", target_type="attr", download=False, transform=transform)
full_test_ds  = CelebA(root=DATA_DIR, split="test",  target_type="attr", download=False, transform=transform)

# Subsampling dataset since I am using CPU
train_dataset = Subset(full_train_ds, list(range(5000)))
test_dataset  = Subset(full_test_ds, list(range(1000)))

# Grabbing the indices for the target and the concepts
target_y_idx = full_train_ds.attr_names.index(TARGET_ATTR)
concept_indices = [full_train_ds.attr_names.index(attr) for attr in CONCEPT_ATTRS]

print(f"Train subset size: {len(train_dataset)} | Test subset size: {len(test_dataset)}")

# Load in a pre-trained ResNet-18 Neural network and remove the final layer.
# Instead, final layer will output a vector of features, which we flatten
# to whatever dimension we need for our problem. Ex. 1 for "Smiling", 10 
# for all features of interest. 
class SharedResNetBackbone(nn.Module):
    """
    Base feature extractor class wrapping an ImageNet-pretrained ResNet-18 backbone.
    Strips off the default 1000-class fc layer, producing a 512-dimensional vector.
    """
    def __init__(self, pretrained=True):
        super(SharedResNetBackbone, self).__init__()
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        resnet = resnet18(weights=weights)
        
        self.in_features = resnet.fc.in_features  # 512 features
        
        # Remove original fc layer by slicing up to AvgPool2d
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])

    def forward(self, x):
        features = self.backbone(x)
        return torch.flatten(features, 1)  # Output shape: (N, 512)

Active execution device: cpu
Train subset size: 5000 | Test subset size: 1000


In [2]:
#Part 1

# Custom collate function to extract 'Smiling' target vector
# This is used because CelebA includes large multidimensional images
# so we define a custom collate_fn to avoid returning all 40 attributes per 
# image and just the specific Smiling target label.
# Returns a tuple of the images and their corresponding target attribute
def baseline_collate_fn(batch):
    images, target_attrs = zip(*batch)
    images = torch.stack(images, dim=0)
    
    # Extract single target 'Smiling' and convert (-1, 1) -> (0.0, 1.0)
    raw_targets = torch.stack([attr[target_y_idx] for attr in target_attrs])
    targets = (raw_targets == 1).float().unsqueeze(1)  # Shape: (N, 1)
    
    return images, targets

# DataLoaders which automate feeding the data into the model.
# Creates a stack of images (size of which is batch_size = 64), where each image is 64x64 pixels with 3 color channels
train_loader_y = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=baseline_collate_fn)
test_loader_y  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=baseline_collate_fn)

# Going to use our pre-defined backbone class so we can 
# re-use the already trained model. Adds on a linear layer 
# so we can use it for our Smiling classification problem
# and produce a single output per image. This is not a probability yet
# because this is done later with BCEWithLogitsLoss() which applies the 
# sigmoid function.
class BaselineClassifier(nn.Module):
    def __init__(self, backbone, num_classes=1):
        super(BaselineClassifier, self).__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(backbone.in_features, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

# Instantiate Shared Backbone and Model
shared_backbone_1 = SharedResNetBackbone(pretrained=True)
baseline_model = BaselineClassifier(backbone=shared_backbone_1, num_classes=1).to(device)

criterion_y = nn.BCEWithLogitsLoss()  #Combines both the sigmoid function wtih the binary cross-entropy function
optimizer_y = optim.Adam(baseline_model.parameters(), lr=LEARNING_RATE)  #Instead of SGD, Adam computes individual learning rate per parameter

# Training Loop
print("\n--- Training Baseline Classifier (x -> y) ---")
for epoch in range(NUM_EPOCHS):
    #Initialize starting point for this epoch
    baseline_model.train()
    running_loss = 0.0

    #Iterate through the batches of images
    for images, targets in train_loader_y:
        images, targets = images.to(device), targets.to(device)

        #Clearing gradient from the previous batch
        optimizer_y.zero_grad()
        
        logits = baseline_model(images) # Forward pass
        loss = criterion_y(logits, targets) # Loss function
        loss.backward() # Backwards pass
        optimizer_y.step() # Update model weights

        # Tracking total loss for this epoch
        running_loss += loss.item() * images.size(0)

    #Finding average loss for this epoch across the dataset
    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}] - Loss: {epoch_loss:.4f}")

# Model Evaluation (Accuracy & AUROC)

# Can use these two functions to track metrics internally rather than doing
# so by hand with a list
test_acc_metric = BinaryAccuracy().to(device) # % of total test images correctly classified as Smiling
test_auroc_metric = BinaryAUROC().to(device)  # Measures how confident the model is in a correct/incorrect answer

# Switch to eval mode so no training behaviors are triggered
baseline_model.eval()

# Iterate through our test images, pass them through the model, apply sigmoid
# function to get a probability of smiling, and assess accuracy across
# entire test dataset
with torch.no_grad(): #Disables gradient calculations since we are evaluating
    for images, targets in test_loader_y:
        images, targets = images.to(device), targets.to(device)
        logits = baseline_model(images)
        probs = torch.sigmoid(logits)
        
        test_acc_metric.update(probs, targets.int())
        test_auroc_metric.update(probs, targets.int())

#Compute and print our total accuracies
print("\n=== Baseline Classifier (x -> y) Report ===")
print(f"Test Accuracy: {test_acc_metric.compute().item() * 100:.2f}%")
print(f"Test AUROC:    {test_auroc_metric.compute().item():.4f}")


--- Training Baseline Classifier (x -> y) ---
Epoch [1/5] - Loss: 0.5238
Epoch [2/5] - Loss: 0.1746
Epoch [3/5] - Loss: 0.0462
Epoch [4/5] - Loss: 0.0164
Epoch [5/5] - Loss: 0.0091

=== Baseline Classifier (x -> y) Report ===
Test Accuracy: 87.00%
Test AUROC:    0.9433


In [4]:
#Part 2

#Similar to previous collate function but instead Extracts the 10 attributes of interest
def concept_collate_fn(batch):
    images, target_attrs = zip(*batch)
    images = torch.stack(images, dim=0)
    
    # Extract all 10 concept values
    raw_concepts = torch.stack([
        torch.tensor([attr[idx] for idx in concept_indices]) 
        for attr in target_attrs
    ])
    concepts = (raw_concepts == 1).float()
    
    return images, concepts

# Instantiate DataLoaders with new collate function
train_loader_c = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=concept_collate_fn)
test_loader_c  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=concept_collate_fn)

# Similar class as previous but now the linear head has 10 attributes instead of 1
class ConceptPredictor(nn.Module):
    def __init__(self, backbone, num_concepts=10):
        super(ConceptPredictor, self).__init__()
        self.backbone = backbone
        self.concept_head = nn.Linear(backbone.in_features, num_concepts)

    def forward(self, x):
        features = self.backbone(x)
        return self.concept_head(features)  # Shape: (N, 10)

# Initialize the model with the shared backbone and 10 attributes of interest
shared_backbone_2 = SharedResNetBackbone(pretrained=True)
concept_model = ConceptPredictor(backbone=shared_backbone_2, num_concepts=NUM_CONCEPTS).to(device)

# Defining the criterion and omptimizer as discussed before 
criterion_c = nn.BCEWithLogitsLoss() # Able to handle multi-label targets across all 10 outputs
optimizer_c = optim.Adam(concept_model.parameters(), lr=LEARNING_RATE)

print("\n--- Training Concept Predictor (x -> c) ---")

#Same steps for the epochs as before, except calculating loss across all 10 attributes
for epoch in range(NUM_EPOCHS):
    concept_model.train()
    running_loss = 0.0
    
    for images, concept_targets in train_loader_c:
        images, concept_targets = images.to(device), concept_targets.to(device)
        
        optimizer_c.zero_grad()
        logits = concept_model(images)
        loss = criterion_c(logits, concept_targets)
        loss.backward()
        optimizer_c.step()
        
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}] - Loss: {epoch_loss:.4f}")


# Setup metrics as before, but will use lists to evaluate each concept individually
acc_metric = BinaryAccuracy().to(device)
f1_metric = BinaryF1Score().to(device)

# Lists to collect predictions and targets per-attribute
all_preds, all_targets = [], []

# Same eval set up as before
concept_model.eval()
with torch.no_grad():
    for images, concept_targets in test_loader_c:
        images, concept_targets = images.to(device), concept_targets.to(device)
        logits = concept_model(images)
        probs = torch.sigmoid(logits)

        #Append the 10 attribute performances to the lists
        all_preds.append(probs)
        all_targets.append(concept_targets.int())

# Concatenate all the batches together so that we have performance across attributes for all batches in matrix form
all_preds = torch.cat(all_preds, dim=0)
all_targets = torch.cat(all_targets, dim=0)

# Now to iterate through each attribute of interest, look at the predict performances, and calculate
# the metrics so we get performance across every attribute for all of the data.
per_concept_accs, per_concept_f1s = [], []

print("\n--- Per-Concept Performance ---")
for i, concept_name in enumerate(CONCEPT_ATTRS):
    c_preds, c_targets = all_preds[:, i], all_targets[:, i]
    
    c_acc = acc_metric(c_preds, c_targets).item()
    c_f1 = f1_metric(c_preds, c_targets).item()
    
    per_concept_accs.append(c_acc)
    per_concept_f1s.append(c_f1)
    
    print(f"{concept_name:<22} | Acc: {c_acc * 100:.2f}% | F1: {c_f1:.4f}")

# Compute macro-average across all 10 attributes
mean_acc = sum(per_concept_accs) / NUM_CONCEPTS
mean_f1  = sum(per_concept_f1s) / NUM_CONCEPTS

print("\n=== Concept Predictor (x -> c) Report ===")
print(f"Mean Concept Accuracy (Macro): {mean_acc * 100:.2f}%")
print(f"Mean Concept F1 Score (Macro): {mean_f1:.4f}")


--- Training Concept Predictor (x -> c) ---
Epoch [1/5] - Loss: 0.5160
Epoch [2/5] - Loss: 0.3483
Epoch [3/5] - Loss: 0.2568
Epoch [4/5] - Loss: 0.1801
Epoch [5/5] - Loss: 0.1208

--- Per-Concept Performance ---
Mouth_Slightly_Open    | Acc: 80.90% | F1: 0.8000
High_Cheekbones        | Acc: 78.30% | F1: 0.7751
Chubby                 | Acc: 93.10% | F1: 0.2247
Narrow_Eyes            | Acc: 84.20% | F1: 0.0482
Bags_Under_Eyes        | Acc: 80.90% | F1: 0.4195
Big_Lips               | Acc: 65.60% | F1: 0.2863
Big_Nose               | Acc: 79.30% | F1: 0.4450
Pointy_Nose            | Acc: 70.50% | F1: 0.3867
Bushy_Eyebrows         | Acc: 87.60% | F1: 0.3922
Arched_Eyebrows        | Acc: 76.50% | F1: 0.5155

=== Concept Predictor (x -> c) Report ===
Mean Concept Accuracy (Macro): 79.69%
Mean Concept F1 Score (Macro): 0.4293


Interpretation of above results: While accuracy seems high across all attributes, the F1 score tells us that the model isn't actually doing a good job classifying, it is just getting lucky that alot of the data in the dataset just doesnt contain images with those attributes present. For example, if it classifies an image as "not chubby", it tends to be correct because only a small subset of the data actually contains images labeled "chubby". The F1 scores basically confirms that the model was well trained on "Mouth Slightly Open" and "High Cheekbones" likely because there are ample examples of these in our training dataset, so the model got good at accurately detecting it. The other attributes however, the model does a poor job of detecting them, but gets lucky due to the proportionaly disparity in the test set of what images actually have what features.

## Part 3: Concept Bottleneck Model
Goal: So far the internals of the model are a black box in terms of how it is deciding whether a person is smiling or not. For this part, we are going to separate the classification problem into two steps. The first being allow the neural network to provide probabilities of the 10 attributes of interest as in part 2. Then, use another layer that takes those ten attributes in as an input, and creates a prediction on whether the person is smiling. Basically forcing some transparency into the model by seeing how certain features become indicative of someone smiling.